# 📗 RunPod GPU와 GCP VM으로 배포하기: GPU 파드의 vLLM과 VM의 서빙 API를 proxy 주소로 잇기

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">내 PC 노트북:</strong> GPU 파드의 vLLM 응답을 확인하고, VM의 서빙 API에 요청을 보냅니다.<br><strong style="font-size:1em;color:#1f3a5f;">내 PC 터미널:</strong> VM용 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 scp로 VM에 올립니다.<br><strong style="font-size:1em;color:#1f3a5f;">GCP VM 터미널:</strong> VS Code Remote-SSH 창에서 git pull과 docker compose로 서빙 API를 실행합니다.<br><strong style="font-size:1em;color:#1f3a5f;">RunPod 콘솔:</strong> GPU를 고르고 GPU 파드를 만들고(Deploy), 로그와 GPU 사용량을 봅니다.<br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 부록 01(GCP VM 준비)의 VM, 교안 00의 serving-deploy 저장소(VM에 clone), 교안 01에서 GHCR에 올린 이미지(latest 태그)가 있어야 합니다.<br><strong style="font-size:1em;color:#1f3a5f;">비용:</strong> GPU 파드는 켜져 있는 동안 시간당 요금이 붙습니다. 교안 03과 과제 02에서도 배포한 API가 이 파드를 부르므로 켜 둔 채 이어 가고, day78의 마지막인 과제 02 끝에서 종료합니다.</div>

#### **두 컴퓨터 서빙 구성(two-machine serving)**

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델을 실행하는 vLLM 서버는 GPU 컴퓨터에, 키 확인과 사용량 기록을 맡는 서빙 API는 GPU 없는 컴퓨터에 두고, 서빙 API가 vLLM의 주소와 키로 요청을 넘기는 배포 구성</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> GPU는 비쌉니다. 모델만 GPU 파드에 두면 서빙 API를 다시 배포해도 GPU 파드는 그대로이고, GPU 파드를 종료해도 키 DB는 남습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 모델 서버 하나를 여러 API가 함께 쓰는 서비스, API는 자주 고치고 모델은 가끔 바꾸는 서비스입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/20_two_machine_serving.png" width="1050" alt="내 PC가 GCP VM의 서빙 API(8080)를 부르고, 서빙 API가 RunPod GPU 파드의 vLLM(8000)을 proxy 주소와 vLLM 키로 부르며, 답변은 vLLM에서 서빙 API를 거쳐 내 PC로 돌아오는 두 컴퓨터 구성">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안에서 만드는 구성입니다. RunPod GPU 파드는 공식 vLLM 이미지로 모델을 실행하고, GCP VM은 compose로 서빙 API 컨테이너를 실행하며, 내 PC 노트북은 VM의 서빙 API에 상담을 요청합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/21_lesson02_goals.png" width="1050" alt="학습 목표 그림: 1 GPU 메모리 계산과 GPU 고르기, 2 RunPod 콘솔에서 GPU 파드를 만들고 로그와 GPU 사용량 보기, 3 VM에서 compose로 서빙 API 실행, 4 볼륨에 남는 키 DB">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day78-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. 모델 크기와 필요한 컨텍스트로 GPU 메모리를 계산하고, RunPod 콘솔에서 조건을 만족하는 가장 싼 GPU를 고릅니다.
2. RunPod 콘솔에서 공식 vLLM 이미지의 GPU 파드를 만들고, 로그와 GPU 사용량을 봅니다.
3. compose 파일로 VM의 서빙 API 컨테이너를 실행하고, GPU 파드의 vLLM에 연결합니다.
4. 컨테이너를 다시 만들어도 볼륨에 둔 키 DB가 남는 것을 확인합니다.

</div>

## 1. 실습 준비

### 1-1. 노트북 준비

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">경로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>, JSONL 읽기 함수와 노무 질문을 한 셀에서 준비합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값 ----
import json
import os
import secrets
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv, set_key

day_dir = Path(".").resolve()       # data, images, output 이 있는 일차 폴더(day78_모델배포_Docker_CICD)
data_dir = day_dir / "data"
output_dir = day_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기 ----
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 노무 질문, 요청 시간 제한 ----
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
REQUEST_TIMEOUT = 120   # 서빙 API 응답을 기다리는 최대 시간(초). VM 의 vLLM 시간 제한 90초보다 길게 둡니다

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>(5절에서 VM 주소로 씀)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER</code>(4-2에서 이미지 이름으로 씀)가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 있는지 이름만 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# .env 에 필요한 값이 있는지 이름만 확인합니다(값은 출력하지 않습니다)
env_check = {}
for name in ["VM_HOST", "GITHUB_OWNER"]:
    env_check[name] = name in os.environ
display(env_check)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">False가 있으면:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 그 값을 적고 커널을 다시 시작합니다.</li>
</ul>
</div>

### 1-2. vLLM 키와 관리자 키 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 파드를 만들 때(3절) vLLM 키를 넣어야 하므로 키부터 만듭니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 키:</strong> GPU 파드의 vLLM이 요청마다 확인하는 키입니다. GPU 파드의 환경 변수(3절)와 VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>(4절)에 같은 값을 넣습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">관리자 키:</strong> VM 서빙 API의 관리자 주소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/admin/keys</code>)에 보내는 키입니다. VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에만 넣습니다.</li>
</ul>
<p style="margin:6px 0;">두 키를 만들어 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 저장합니다(같은 이름의 줄이 있으면 새 값으로 바뀝니다).</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 추측할 수 없는 무작위 문자열로 vLLM 키와 관리자 키를 만듭니다(앞에 용도를 붙입니다)
new_keys = {
    "VLLM_API_KEY": "enkoa-vllm-" + secrets.token_urlsafe(32),
    "ADMIN_API_KEY": "enkoa-admin-" + secrets.token_urlsafe(32),
}

# 실습자료/.env 에 저장하고 다시 읽습니다(키 값은 출력하지 않습니다)
for name, value in new_keys.items():
    set_key(env_path, name, value, quote_mode="never")   # 따옴표 없이 저장해 RunPod 화면에 붙여 넣을 때 따옴표가 들어가지 않게 합니다
load_dotenv(env_path, override=True)

# 저장된 값의 글자 수만 봅니다
saved_lengths = {}
for name in new_keys:
    saved_lengths[name + " 글자 수"] = len(os.environ[name])
display(saved_lengths)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 셀은 한 번만 실행합니다:</strong> 다시 실행하면 키가 바뀌어 3절 파드의 vLLM 키와 달라집니다.</li>
</ul>
</div>

## 2. GPU 고르기

### 2-1. 필요한 GPU 메모리

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 카드는 메모리 크기로 고르므로, 먼저 이 모델을 서빙하는 데 필요한 GPU 메모리를 계산합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">필요한 GPU 메모리:</strong> vLLM이 GPU에 올려 두는 가중치, KV 캐시, 실행 공간(중간 계산 등)을 더한 양입니다. 이 값보다 메모리가 큰 카드를 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 요청마다 앞 토큰들의 K, V 값을 저장해 두는 공간으로, 최대 길이와 동시 요청 수에 비례해 커집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 공간:</strong> 계산 도중의 중간 값, CUDA 그래프, 라이브러리가 쓰는 메모리입니다. 미리 정확히 계산하기 어려워 가중치의 약 20%로 어림합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">BF16(bfloat16, 16비트 부동소수점):</strong> 수 하나를 2바이트로 저장하는 형식입니다. 이 모델의 가중치와 KV 캐시가 이 형식입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">전체 Attention 층과 선형 Attention 층:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>의 32층 가운데 8층만 앞의 모든 토큰의 K, V를 저장하는 전체 Attention 층입니다. 나머지는 토큰 수와 상관없이 크기가 정해진 상태 하나를 두는 선형 Attention(linear attention) 층입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code>:</strong> vLLM이 GPU 전체 메모리 가운데 쓰는 비율입니다. 가중치, 실행 공간, KV 캐시가 모두 이 안에 들어가야 하므로, 필요한 양을 이 비율로 나눈 값이 카드 메모리 이하여야 합니다. vLLM 0.27.0의 기본값은 0.92이고, 이 계산은 0.9로 여유를 둡니다.</li>
</ul>
</div>

<div class="day78-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">가중치</strong> = 파라미터 수 × 2바이트<br><strong style="color:#1f3a5f;">토큰 하나의 KV 캐시</strong> = 2(K, V) × 전체 Attention 층 수 × KV 헤드 수 × 헤드 차원 × 2바이트<br><strong style="color:#1f3a5f;">KV 캐시</strong> = 토큰 하나의 KV 캐시 × 최대 길이 × 동시 요청 수<br><strong style="color:#1f3a5f;">실행 공간</strong> ≈ 가중치 × 0.2<br><strong style="color:#1f3a5f;">필요한 GPU 메모리</strong> = (가중치 + KV 캐시 + 실행 공간) ÷ 0.9</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">선형 Attention 층의 고정 상태는 식에 넣지 않았으므로 계산값은 근사입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/22_gpu_sizing.png" width="1050" alt="필요한 GPU 메모리 계산: 가중치, KV 캐시, 실행 공간을 더해 0.9로 나눈 값과 GPU 카드의 메모리 비교">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>를 최대 길이 8,192토큰(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code>), 동시 요청 16으로 서빙하는 경우입니다. 노무 상담 한 건은 입력 약 5,000토큰, 답 최대 256토큰이라 8,192토큰에 들어갑니다. KV 캐시는 16건이 모두 최대 길이를 쓰는 경우로 계산합니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가중치:</strong> 4,659,865,088 × 2바이트 = 9.3GB(1GB = 10억 바이트)</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">토큰 하나의 KV 캐시:</strong> 2 × 8 × 4 × 256 × 2바이트 = 32,768바이트</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 32,768바이트 × 8,192 × 16 = 4.3GB</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 공간:</strong> 9.3 × 0.2 = 1.9GB</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">필요한 GPU 메모리:</strong> (9.3 + 4.3 + 1.9) ÷ 0.9 = 17.2GB</li>
</ol>
<p style="margin:6px 0;">먼저 Hugging Face에서 이 모델의 설정 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">config.json</code>)과 모델 정보를 읽어, 위 계산에 쓰는 값을 꺼냅니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# Hugging Face 에서 모델 설정 파일(config.json)과 모델 정보(파라미터 수가 들어 있음)를 읽습니다
config_response = httpx.get("https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json", timeout=30)
text_config = config_response.json()["text_config"]   # 이 모델은 이미지 입력도 받아, 글을 처리하는 부분의 설정이 text_config 안에 있습니다
model_info = httpx.get("https://huggingface.co/api/models/Qwen/Qwen3.5-4B", timeout=30).json()

# 계산에 쓸 값을 꺼냅니다
params = model_info["safetensors"]["total"]                                 # 전체 파라미터 수
full_attention_layers = text_config["layer_types"].count("full_attention")   # 층마다 적힌 종류 가운데 전체 Attention 층의 수
kv_heads = text_config["num_key_value_heads"]                                # KV 헤드 수
head_dim = text_config["head_dim"]                                           # 헤드 하나의 차원

model_config = {
    "층 수(num_hidden_layers)": text_config["num_hidden_layers"],
    "전체 Attention 층 수": full_attention_layers,
    "KV 헤드 수(num_key_value_heads)": kv_heads,
    "헤드 차원(head_dim)": head_dim,
    "가중치 형식(dtype)": text_config["dtype"],
    "파라미터 수": params,
}
display(model_config)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">층 수 32, 전체 Attention 층 수 8:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">layer_types</code>에 층마다 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">full_attention</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">linear_attention</code>이 적혀 있고, 전체 Attention 층만 KV 캐시를 토큰마다 쌓습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가중치 형식 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">bfloat16</code>:</strong> 파라미터 하나가 2바이트입니다.</li>
</ul>
<p style="margin:6px 0;">읽은 값으로 위 계산을 코드로 합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 서빙 조건: 최대 길이는 컨테이너 시작 명령의 --max-model-len, 동시 요청은 서빙 벤치마킹 단원의 기본 부하
max_len = 8192
concurrency = 16

GB = 10**9   # 1GB 는 10억 바이트
WORKING_SPACE_RATIO = 0.2   # 실행 공간(중간 계산, CUDA 그래프, 라이브러리)은 가중치의 약 20% 로 어림합니다
UTILIZATION = 0.9    # vLLM 이 GPU 메모리에서 쓰는 비율(기본 0.92 보다 조금 낮춰 여유를 둡니다)

# 위의 다섯 식을 차례로 계산합니다(BF16 은 수 하나에 2바이트, 앞의 2 는 K 와 V)
weights_gb = params * 2 / GB
kv_bytes_per_token = 2 * full_attention_layers * kv_heads * head_dim * 2
kv_cache_gb = kv_bytes_per_token * max_len * concurrency / GB
working_space_gb = weights_gb * WORKING_SPACE_RATIO
need_gb = (weights_gb + kv_cache_gb + working_space_gb) / UTILIZATION

sizing = {
    "가중치(GB)": round(weights_gb, 1),
    "토큰 하나의 KV 캐시(바이트)": kv_bytes_per_token,
    "KV 캐시(GB)": round(kv_cache_gb, 1),
    "실행 공간(GB)": round(working_space_gb, 1),
    "필요한 GPU 메모리(GB)": round(need_gb, 1),
}
display(sizing)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가중치는 넉넉하게 잡은 값:</strong> 이미지 입력 층을 올리지 않으므로(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code>) 실제로 GPU에 올라가는 가중치는 약 8 GiB입니다.</li>
</ul>
</div>

### 2-2. RunPod 콘솔에서 GPU 고르기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">계산한 메모리를 기준으로, Deploy 화면의 GPU 카드를 세 조건으로 거른 뒤 가장 싼 카드를 고릅니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">메모리:</strong> 2-1의 17.2GB 이상입니다. Filter GPUs 창의 Minimum VRAM 칸은 그대로 두고(아래 화면에서 0 GB), 카드에 적힌 VRAM을 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Secure Cloud, CUDA 13.0 이상:</strong> 공식 vLLM 이미지가 CUDA(NVIDIA GPU로 계산하는 소프트웨어) 13을 쓰므로, 머신의 GPU 드라이버가 CUDA 13을 지원해야 합니다. RunPod 데이터센터(Secure Cloud) 머신 가운데 CUDA 13.0 이상만 남깁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">재고:</strong> 지금 빌릴 수 있는 카드입니다. Available 탭에서 봅니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">RunPod 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.runpod.io/pods</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">Filter를 누르고 CUDA versions에서 12.x를 모두 해제해 13.0 이상만 남깁니다.</li>
<li style="margin:3px 0;">Cloud type이 Secure인지 보고 Apply filters를 누릅니다.</li>
<li style="margin:3px 0;">GPU 목록 위의 정렬 상자를 Price로 바꾸고 Available 탭을 봅니다. 비싼 카드부터 나오므로 맨 아래가 가장 쌉니다.</li>
<li style="margin:3px 0;">맨 아래쪽에서 메모리가 17.2GB 이상인 가장 싼 카드를 누릅니다. Deploy Pod는 3절에서 누릅니다.</li>
</ol>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · Pods &gt; Deploy 화면(GPU 파드 만들기)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Compute의 GPU 탭에서 Filter를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Filter 버튼</li></ul></div>
<img src="images/runpod_01_deploy_page.png" width="1050" alt="Pods > Deploy 화면(GPU 파드 만들기). 빨간 상자: Filter 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · Filter GPUs &gt; CUDA versions 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 13.0 이상만 선택한 채로 두고 12.x는 모두 해제합니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 선택된 13.0~13.4</li></ul></div>
<img src="images/runpod_02_cuda_versions.png" width="1050" alt="Filter GPUs > CUDA versions 목록. 빨간 상자: 선택된 13.0~13.4">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · Filter GPUs 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Cloud type이 Secure인지 보고, CUDA versions를 고친 뒤 Apply filters를 누릅니다. Secure는 RunPod가 검증한 데이터센터의 GPU(Community는 개인이 빌려주는 GPU)라, 키를 넣어 외부에 여는 서버를 더 안정적이고 안전하게 띄웁니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Cloud type(Secure), CUDA versions, Apply filters</li></ul></div>
<img src="images/runpod_03_filter_dialog.png" width="1050" alt="Filter GPUs 창. 빨간 상자: Cloud type(Secure), CUDA versions, Apply filters">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · GPU 카드 목록(Price 정렬, Available)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Price 정렬은 비싼 카드부터 나오므로, 목록 맨 아래에서 메모리 17.2GB 이상인 가장 싼 카드를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 고른 GPU 카드(RTX PRO 4000, <span>&#36;</span>0.57/hr, 24GB)</li></ul></div>
<img src="images/runpod_04_gpu_card.png" width="1050" alt="GPU 카드 목록(Price 정렬, Available). 빨간 상자: 고른 GPU 카드(RTX PRO 4000, &#36;0.57/hr, 24GB)">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가격과 재고는 볼 때마다 바뀝니다:</strong> 메모리 조건만 넘으면 어느 카드든 됩니다.</li>
</ul>
</div>

## 3. GPU 파드 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">2-2에서 고른 GPU로, 공식 vLLM 이미지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm/vllm-openai:v0.27.0</code>를 실행하는 GPU 파드를 하나 만듭니다. 이 파드의 vLLM이 모델을 실행하고, 1-2의 vLLM 키가 든 요청만 받습니다.</p>
</div>

### 3-1. 템플릿과 Set overrides

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">템플릿 카드</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">템플릿(template):</strong> 파드 설정을 저장해 둔 묶음입니다. 컨테이너 이미지, 시작 명령, 디스크 크기, 공개 포트, 환경 변수가 들어 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">쓰는 때:</strong> RunPod가 준비해 둔 환경(PyTorch 등)으로 바로 시작할 때 고릅니다. 같은 설정의 파드를 자주 만들면 Templates 메뉴에 내 템플릿으로 저장해 두고 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">템플릿 카드:</strong> Deploy 화면 Workload에 지금 선택된 템플릿이 카드로 보입니다(아래 화면에서는 PyTorch 템플릿).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Set overrides:</strong> 카드의 이 버튼으로 바꾼 값은 이 파드에만 들어갑니다. 저장된 템플릿은 바뀌지 않습니다.</li>
</ul>
<p style="margin:6px 0;">이 교안은 템플릿을 새로 고르지 않고, 선택된 템플릿 카드의 Set overrides에서 값을 바꿔 vLLM 파드로 만듭니다. 넣을 값은 3-2의 화면마다 설명합니다.</p>
</div>

### 3-2. 파드 만들기와 파드 id

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Deploy Pod는 한 번만 누릅니다.</strong> 또 누르면 요금이 붙는 파드가 하나 더 생깁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">RunPod 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.runpod.io/pods</code></li>
</ul>
<p style="margin:6px 0;">2-2에서 고른 GPU 카드를 누른 화면에서 시작합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · GPU count 와 Instance pricing</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> GPU count는 1, 요금 방식은 On-Demand를 고릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> GPU count 1, On-Demand</li></ul></div>
<img src="images/runpod_05_gpu_count_pricing.png" width="1050" alt="GPU count 와 Instance pricing. 빨간 상자: GPU count 1, On-Demand">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · Workload 의 템플릿 카드</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 템플릿은 그대로 두고 Set overrides를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Set overrides 버튼</li></ul></div>
<img src="images/runpod_06_set_overrides.png" width="1050" alt="Workload 의 템플릿 카드. 빨간 상자: Set overrides 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · Edit template overrides 창(위쪽)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Container image, Container start command(+ Start command로 칸 열기), Container disk 50을 넣고 Jupyter, SSH 체크를 끕니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Container image, Container start command, 체크 두 개, Container disk</li></ul></div>
<img src="images/runpod_07_overrides_top.png" width="1050" alt="Edit template overrides 창(위쪽). 빨간 상자: Container image, Container start command, 체크 두 개, Container disk">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">3번 화면에 넣을 값입니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">칸</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">넣을 값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이유</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Container image</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm/vllm-openai:v0.27.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 서버가 든 공식 이미지입니다(CUDA 13.0.3 기준).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Container start command</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">아래 한 줄</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM에 넘길 모델과 옵션입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Container disk</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">50 GB</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모델 파일(약 9.3GB)을 받아 두는 공간입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Start Jupyter notebook, SSH terminal access</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">두 체크 모두 끔</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 파드는 vLLM만 실행합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">공식 vLLM 이미지는 켜질 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code>를 먼저 실행합니다(ENTRYPOINT). 시작 명령 칸의 한 줄은 그 뒤에 붙으므로 모델 이름과 옵션만 적습니다. 아래 한 줄을 그대로 붙여 넣습니다.</p>
</div>

```text
Qwen/Qwen3.5-4B --served-model-name qwen3.5-4b --max-model-len 8192 --language-model-only --reasoning-parser qwen3 --enable-prompt-tokens-details --enable-prefix-caching --host 0.0.0.0 --port 8000
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">띄울 모델의 Hugging Face 저장소 이름입니다. 켜질 때 모델 파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HF_HOME</code> 폴더로 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name qwen3.5-4b</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">API 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code> 값으로 쓸 이름입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>에도 나옵니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 8192</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">한 요청에서 입력과 출력을 합친 최대 토큰 수입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지, 영상 입력을 끄고 텍스트만 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser qwen3</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">답의 생각 블록을 떼어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">message.reasoning</code>에 담습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prompt-tokens-details</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">토큰 사용량에 캐시에서 다시 쓴 입력 토큰 수를 넣습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">같은 앞부분의 계산 결과를 다음 요청에서 다시 씁니다. vLLM 0.27.0은 Qwen3.5 같은 하이브리드 모델에서 기본으로 꺼 두므로 직접 켭니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 0.0.0.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너 밖(proxy 주소)에서 온 요청도 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port 8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM이 요청을 받는 포트로, 다음 화면의 HTTP 포트와 같아야 합니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · Edit template overrides 창(아래쪽)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> HTTP 포트를 8000 하나로 바꾸고 TCP 22는 지웁니다. 환경 변수 VLLM_API_KEY, HF_HOME을 넣은 뒤 Set overrides를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Exposed HTTP ports 8000, Environment variables 두 줄, Set overrides</li></ul></div>
<img src="images/runpod_08_overrides_bottom.png" width="1050" alt="Edit template overrides 창(아래쪽). 빨간 상자: Exposed HTTP ports 8000, Environment variables 두 줄, Set overrides">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">4번 화면에 넣을 값입니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">칸</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">넣을 값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이유</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Exposed HTTP ports</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8000</code> 하나</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 포트입니다. 템플릿 기본의 8888은 지웁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Exposed TCP ports</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">비움</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">템플릿 기본의 22(SSH)를 지웁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">환경 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1-2에서 만든 키</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VS Code에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>를 열어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY=</code> 뒤의 값을 복사합니다. vLLM은 이 환경 변수를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code>처럼 읽어, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> 주소(상담, 모델 목록)에는 이 키가 든 요청만 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">환경 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HF_HOME</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/workspace/hf</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모델 파일을 받아 둘 폴더입니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">파드 환경 변수는 암호화되지 않아 팀원이 볼 수 있으므로, 이 키는 다른 곳에 쓰지 않습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 5 · Pod name 과 오른쪽 요약</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Pod name에 serving-vllm을 적고 요약의 이미지와 시간당 요금을 확인한 뒤 Deploy Pod를 한 번만 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Pod name, Total / hour, Deploy Pod</li></ul></div>
<img src="images/runpod_10_name_deploy.png" width="1050" alt="Pod name 과 오른쪽 요약. 빨간 상자: Pod name, Total / hour, Deploy Pod">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">요약의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Nothing mounted at the template's path</code> 안내는 그대로 둡니다. RunPod 볼륨(Persistent storage)은 요금이 따로 붙어 붙이지 않고, 모델 파일은 새 파드마다 다시 받습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Instance not available</code>이 뜨면 그 GPU의 재고가 없는 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Deploy when available</code> 창은 닫고, 2-2 화면 4에서 그다음으로 싼 카드를 누른 뒤 이 절의 화면 1~5를 다시 합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 6 · Pods &gt; serving-vllm 상세(Connect 탭, 이미지 받는 중)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 파드 이름 아래 ID를 복사해 실습자료/.env의 GPU_POD_ID에 적습니다. 처음에는 이미지(약 9GB)를 받는 진행 막대가 보입니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 파드 ID</li></ul></div>
<img src="images/runpod_11_pod_id.png" width="1050" alt="Pods > serving-vllm 상세(Connect 탭, 이미지 받는 중). 빨간 상자: 파드 ID">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">복사한 파드 id를 VS Code에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 한 줄로 적습니다(같은 이름의 줄이 있으면 값만 바꿉니다).</p>
</div>

```text
GPU_POD_ID=<복사한 파드 id>
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 7 · serving-vllm &gt; Details</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Image와 Start command가 위에서 넣은 값인지 확인하고, Pricing의 Total(시간당 요금)을 봅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Pricing 제목, Image, Template, Start command</li></ul></div>
<img src="images/runpod_15_details.png" width="1050" alt="serving-vllm > Details. 빨간 상자: Pricing 제목, Image, Template, Start command">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod proxy 주소</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">RunPod가 파드에서 공개한 HTTP 포트마다 만드는 인터넷 주소로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://&lt;파드 id&gt;-&lt;포트&gt;.proxy.runpod.net</code>으로 온 요청을 파드의 그 포트로 넘깁니다.</li>
<li style="margin:3px 0;">내 PC 노트북과 VM의 서빙 API는 이 주소로 파드의 vLLM을 부릅니다. vLLM은 8000번 포트이고 OpenAI 호환 API는 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code>을 붙이므로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://&lt;파드 id&gt;-8000.proxy.runpod.net/v1</code>입니다.</li>
</ul>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 다시 읽어 이 주소를 만듭니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
load_dotenv(env_path, override=True)   # 콘솔에서 복사해 .env 에 적은 GPU_POD_ID 를 다시 읽습니다

# GPU 파드의 vLLM 주소(RunPod proxy)와 vLLM 키 헤더(vLLM 은 키를 Authorization 헤더에 'Bearer 키' 모양으로 받습니다)
VLLM_URL = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1"
vllm_headers = {"Authorization": "Bearer " + os.environ["VLLM_API_KEY"]}
print("vLLM 주소:", VLLM_URL)

## 4. VM에서 compose로 서빙 API 실행

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 파드가 모델을 받아 준비되기까지 약 10분이 걸리므로, 그동안 VM에 서빙 API를 띄웁니다. 파드가 준비되었는지는 5-1에서 확인합니다.</p>
</div>

### 4-1. 실행 설정 파일 compose.yaml

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Docker Compose</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">컨테이너를 띄울 때 붙이는 옵션(이미지, 포트, 환경 변수, 볼륨)을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compose.yaml</code> 파일에 적어 두고, 그 파일이 있는 폴더에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose up -d</code> 한 줄로 컨테이너를 띄우는 Docker 도구입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 교안 01에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code>에 옵션을 길게 붙여 컨테이너를 띄웠습니다. 다시 띄울 때마다 같은 옵션을 빠짐없이 입력해야 합니다. compose는 옵션을 파일로 남겨 두므로 몇 번을 다시 띄워도 같은 설정으로 뜹니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Dockerfile과 다른 점:</strong> Dockerfile은 이미지에 무엇을 넣을지, compose 파일은 그 이미지를 어떤 옵션으로 실행할지 적습니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/25_compose_concept.png" width="1050" alt="Docker Compose 그림: compose.yaml에 적은 image, ports, env_file, volumes, restart 다섯 설정을 docker compose up -d 한 줄로 읽어 GCP VM에 serving-api 컨테이너를 띄움, 컨테이너는 8080 포트로 요청을 받고 같은 폴더 .env의 값을 환경 변수로 받으며 service-data 볼륨에 키 DB를 둠">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM에서 서빙 API를 띄우는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/compose.yaml</code>입니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · deploy/compose.yaml</strong></div>

```yaml
# GCP VM 에서 서빙 API 컨테이너를 띄우는 실행 설정(교안 02)
# 실행: 이 deploy 폴더에서 docker compose up -d, 끄기: docker compose down
# 같은 폴더의 .env 를 두 번 쓴다. ${...} 자리는 compose 가 채우고(이미지 이름), env_file 은 컨테이너 환경 변수가 된다

name: serving-deploy                                  # compose 프로젝트 이름. 정하지 않으면 폴더 이름(deploy)이 되어 같은 VM 의 다른 저장소 compose 와 겹친다

# 띄울 컨테이너(서비스) 목록
services:
  serving-api:                                        # 서비스 이름. compose 가 컨테이너 이름을 serving-deploy-serving-api-1 로 만든다
    image: ${SERVING_IMAGE}:${IMAGE_TAG:-latest}      # 띄울 이미지. ${이름} 은 명령 앞의 환경 변수나 같은 폴더 .env 에서 채우고, ${IMAGE_TAG:-latest} 는 IMAGE_TAG 가 없거나 비면 latest 가 된다(배포 작업은 커밋 sha 를 준다)
    # VM 포트를 컨테이너 포트에 잇는 목록(docker run -p 와 같다)
    ports:
      - "8080:8080"                                   # VM 의 8080 포트를 컨테이너의 8080 포트에 잇는다("VM 쪽:컨테이너 쪽")
    env_file: .env                                    # 같은 폴더 .env 의 이름=값 줄을 모두 컨테이너 환경 변수로 넣는다(docker run -e 를 여러 번 붙인 것과 같다). vLLM 주소, vLLM 키, 관리자 키, 시간 제한을 이미지에 넣지 않고 여기서 준다
    # 컨테이너 환경 변수를 하나씩 더 적는다
    environment:
      IMAGE_TAG: ${IMAGE_TAG:-latest}                 # 지금 띄운 이미지 태그를 컨테이너 안에도 IMAGE_TAG 로 넘긴다(배포 버전 표시용)
    # 컨테이너에 붙일 볼륨 목록(docker run -v 와 같다)
    volumes:
      - service-data:/app/data                        # 이름 있는 볼륨 service-data 를 키 DB 폴더 /app/data 에 붙인다. 컨테이너를 지우고 다시 만들어도 볼륨의 파일(키 DB)은 남는다
    restart: unless-stopped                           # 서버가 죽거나 VM 이 다시 켜지면 저절로 다시 띄운다(사람이 docker compose stop 으로 멈춘 경우는 빼고)

# 이 파일이 쓰는 이름 있는 볼륨을 한 번 선언한다
volumes:
  service-data:                                       # 위 서비스가 /app/data 에 붙이는 볼륨. Docker 에는 serving-deploy_service-data 라는 이름으로 만들어진다
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">YAML은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름: 값</code>으로 설정을 적고, 들여쓰기(공백 2칸)로 어느 설정에 속하는지 나타냅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">#</code> 뒤는 주석입니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">줄</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">name: serving-deploy</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">compose 프로젝트 이름입니다. 컨테이너와 볼륨 이름 앞에 붙습니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy-serving-api-1</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy_service-data</code>).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services:</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">띄울 컨테이너(서비스) 목록을 시작합니다. 아래에 들여 쓴 줄이 그 안의 설정입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api:</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서비스 이름입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose logs serving-api</code>처럼 명령에서 이 이름으로 가리킵니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image: &#36;{SERVING_IMAGE}:&#36;{IMAGE_TAG:-latest}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">띄울 이미지입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{이름}</code>은 그 이름의 값으로 바뀌는 자리이고, 값은 명령 앞에 붙인 환경 변수나 같은 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 가져옵니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVING_IMAGE</code>는 4-2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 적습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{IMAGE_TAG:-latest}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{이름:-기본값}</code> 모양으로, 값이 없을 때 쓸 기본값을 정합니다. 이름 뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">:-</code>(콜론과 빼기 기호 두 글자가 한 기호)는 &quot;이 이름의 값이 없거나 비어 있으면 뒤의 값을 쓴다&quot;는 표시이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code> 뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>가 그 기본값입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>를 주지 않으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;아이디&gt;/serving-deploy:latest</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG=3f2a9c...</code>처럼 주면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;아이디&gt;/serving-deploy:3f2a9c...</code>가 뜹니다. 이 교안은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>를 주지 않아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>가 뜨고, 교안 03의 자동 배포는 커밋 sha(커밋마다 다른 40자리 고유 번호, 교안 01 4-2)를 줍니다. 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ports</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>(목록 항목 표시)와는 다른 기호입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ports:</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">- &quot;8080:8080&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>로 시작하는 줄은 목록의 한 항목입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;VM 포트:컨테이너 포트&quot;</code>로 VM의 8080을 컨테이너의 8080에 잇습니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -p 8080:8080</code>과 같습니다). 따옴표로 감싸 글자 그대로 읽게 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env_file: .env</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">같은 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름=값</code> 줄을 모두 컨테이너의 환경 변수로 넣습니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run --env-file .env</code>와 같습니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">environment:</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG: &#36;{IMAGE_TAG:-latest}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">환경 변수를 하나씩 더 넣습니다. 지금 띄운 이미지 태그를 컨테이너 안에서도 알 수 있게 넘깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">volumes:</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">- service-data:/app/data</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이름 있는 볼륨(Docker가 관리하는 이름 붙은 저장 공간) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">service-data</code>를 컨테이너의 키 DB 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app/data</code>에 붙입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -v</code>와 같습니다). 컨테이너를 지우고 다시 만들어도 발급한 키와 사용량 기록이 남습니다(6절).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">restart: unless-stopped</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너가 죽거나 VM이 다시 켜지면 저절로 다시 띄웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose stop</code>으로 직접 멈춘 경우는 다시 띄우지 않습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">맨 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">volumes:</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">service-data:</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 파일이 쓰는 이름 있는 볼륨을 선언합니다. 처음 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose up</code> 할 때 만들어집니다.</td>
</tr>
</tbody>
</table>
</div>

### 4-2. VM에서 쓸 .env 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">compose는 같은 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 값을 읽습니다. 키가 들어 있어 git에 올리지 않으므로, 노트북이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_serving.env</code>로 만들고 4-3에서 VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/.env</code>로 복사합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이름</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">값</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVING_IMAGE</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">띄울 이미지 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;GitHub 아이디&gt;/serving-deploy</code>)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GPU 파드의 vLLM 주소(3-2의 proxy 주소)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 키(1-2)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">관리자 키(1-2)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 답을 기다리는 최대 초. RunPod proxy가 100초에 연결을 끊으므로 그보다 짧은 90초로 둡니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
load_dotenv(env_path, override=True)   # .env 를 다시 읽습니다(7-2 에서 GPU_POD_ID 를 바꾼 뒤 이 셀만 다시 실행해도 새 값이 들어갑니다)

# VM 의 서빙 API 가 쓸 다섯 값을 만듭니다(GHCR 이미지 이름은 소문자)
serving_image = "ghcr.io/" + os.environ["GITHUB_OWNER"].lower() + "/serving-deploy"
vllm_base_url = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1"
vm_env_lines = [
    "SERVING_IMAGE=" + serving_image,
    "VLLM_BASE_URL=" + vllm_base_url,
    "VLLM_API_KEY=" + os.environ["VLLM_API_KEY"],
    "ADMIN_API_KEY=" + os.environ["ADMIN_API_KEY"],
    "VLLM_TIMEOUT_SECONDS=90",
]

# output/vm_serving.env 에 한 줄에 하나씩 씁니다
vm_env_path = output_dir / "vm_serving.env"
vm_env_path.write_text("\n".join(vm_env_lines) + "\n", encoding="utf-8")

# 키 값은 출력하지 않고, 쓴 이름만 확인합니다
written_names = []
for line in vm_env_lines:
    written_names.append(line.split("=")[0])
display(written_names)

### 4-3. .env를 VM에 올리기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키가 든 파일이라 git 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scp</code>로 VM에 바로 복사합니다. <strong style="font-size:1.13em;color:#1f3a5f;">실습자료 폴더</strong>에서 연 내 PC 터미널에서 실행합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
scp day78_모델배포_Docker_CICD/output/vm_serving.env serving-vm:~/serving-deploy/deploy/.env
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scp 보낼곳 받을곳</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">SSH 연결로 내 PC의 파일을 서버로 복사합니다. 앞이 보낼 파일, 뒤가 받을 자리입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm:~/serving-deploy/deploy/.env</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">받을 자리(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Host 이름:경로</code>)입니다. VM에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>라는 이름으로 저장됩니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">복사가 끝나면 줄 끝에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">100%</code>가 보입니다.</p>
</div>

### 4-4. VM에서 compose로 실행

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">저장소를 최신으로 맞추고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>가 올라왔는지 확인한 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy</code> 폴더에서 compose로 서빙 API를 띄웁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/serving-deploy
pwd   # 끝이 serving-deploy이면 맞습니다
git pull

cd deploy
grep -c "=" .env
docker compose up -d
docker compose ps
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 폴더</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pwd</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 폴더로 이동하고, 지금 폴더의 경로를 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git pull</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GitHub의 새 커밋을 받아 VM의 저장소를 최신으로 맞춥니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">grep -c &quot;=&quot; .env</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">=</code>가 든 줄의 수만 출력합니다(값은 보이지 않습니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose up -d</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compose.yaml</code>대로 없는 이미지를 받고 네트워크, 볼륨, 컨테이너를 만들어 실행합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-d</code>는 백그라운드 실행입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose ps</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 compose 프로젝트의 컨테이너 목록과 상태를 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">grep -c</code>가 5, STATUS가 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Up</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>가 올라왔고 서빙 API가 떴습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">denied</code>:</strong> GHCR 이미지를 Public으로 바꿉니다(교안 01 5절).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">port is already allocated</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker rm -f serving-api</code>로 교안 01의 컨테이너를 지우고 다시 실행합니다.</li>
</ul>
<p style="margin:6px 0;">서버가 뜨는 로그를 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
docker compose logs --tail 20
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose logs</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 compose 프로젝트의 컨테이너가 출력한 로그를 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--tail 20</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너마다 로그의 마지막 20줄만 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>:</strong> 서빙 API가 떴습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field required</code>나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY 가 없습니다</code>가 보이면 4-3부터 다시 합니다.</li>
</ul>
</div>

## 5. PC에서 VM 호출

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM의 서빙 API는 GPU 파드의 vLLM이 떠 있어야 답합니다. vLLM이 준비되기를 기다린 뒤(5-1), 내 PC에서 VM의 서빙 API를 부릅니다(5-2~5-4).</p>
</div>

### 5-1. vLLM이 응답할 때까지 기다리기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 파드는 이미지와 모델 파일을 받아 GPU에 올린 뒤에야 요청을 받습니다(Deploy 뒤 약 10분). 이 셀은 내 PC에서 파드의 vLLM 모델 목록(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>)을 30초마다 요청하고, 200이 오면 멈춥니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">상태 코드</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">뜻</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">404</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지를 받는 중입니다(컨테이너가 아직 없음).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">502</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM이 모델을 받아 GPU에 올리는 중입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">401</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM은 떴지만 키가 다릅니다. 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 값이 같은지 봅니다(1-2를 다시 실행했으면 키가 바뀝니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">200</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM이 요청을 받습니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
import time

# 30초마다 모델 목록을 요청하고, 200 이면 멈춥니다(최대 40번, 약 20분)
for attempt in range(40):
    models_response = httpx.get(VLLM_URL + "/models", headers=vllm_headers, timeout=30)
    print(attempt + 1, "번째 요청:", models_response.status_code)
    if models_response.status_code == 200:
        break
    time.sleep(30)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">40번 동안 200이 아니면:</strong> 셀을 다시 실행합니다. 파드의 Logs 탭(Container)에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Error</code>가 있으면 3-2에서 넣은 값을 확인하고 파드를 새로 만듭니다.</li>
</ul>
</div>

### 5-2. 상태 확인과 모델 목록

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC에서 VM의 서빙 API에 두 주소를 요청합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>로 서빙 API가 떠 있는지, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>로 서빙 API가 GPU 파드의 vLLM까지 닿는지 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
VM_URL = "http://" + os.environ["VM_HOST"] + ":8080"   # VM 의 서빙 API 주소(방화벽 규칙으로 연 8080 포트)

# 서빙 API 만 확인하는 주소와, GPU 파드의 vLLM 까지 거치는 주소를 요청합니다
health_response = httpx.get(VM_URL + "/healthz", timeout=REQUEST_TIMEOUT)
vm_models_response = httpx.get(VM_URL + "/v1/models", timeout=REQUEST_TIMEOUT)

vm_check = pd.DataFrame([
    {"주소": "/healthz", "상태 코드": health_response.status_code, "본문": health_response.json()},
    {"주소": "/v1/models", "상태 코드": vm_models_response.status_code, "본문": vm_models_response.json()},
])
display(vm_check)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">두 행 모두 200:</strong> VM의 서빙 API가 GPU 파드의 vLLM까지 닿았습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code> 본문에 서빙 중인 모델 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code>)이 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/v1/models</code>가 502:</strong> GPU 파드가 Running인지, 4-2를 3-2 뒤에 실행했는지, VM <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>가 파드의 값과 같은지 확인합니다.</li>
</ul>
</div>

### 5-3. 서비스 키 발급

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM의 서빙 API에 관리자 키로 요청해 노무 상담 앱(pro 요금제)의 서비스 키를 하나 발급받습니다. 5-4의 상담 요청에 이 키를 씁니다. 키 원문은 출력하지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">issued</code> 변수에만 담습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
ADMIN_HEADERS = {"X-Admin-Key": os.environ["ADMIN_API_KEY"]}   # 관리자 키를 X-Admin-Key 헤더에 담습니다(키 값은 출력하지 않습니다)

# 노무 상담 앱에 pro 요금제 키를 발급합니다
issue_response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": "노무 상담 앱", "plan": "pro"}, timeout=REQUEST_TIMEOUT)
issued = issue_response.json()

# 키 원문(api_key)은 빼고 상태 코드와 key_id, user, plan 을 봅니다
issued_view = {"상태 코드": issue_response.status_code, "key_id": issued["key_id"], "user": issued["user"], "plan": issued["plan"]}
display(issued_view)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상태 코드 201이면 발급된 것입니다. 403이면 노트북의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>가 VM <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 값과 다릅니다.</p>
</div>

### 5-4. 노무 상담 한 건

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">5-3에서 발급한 서비스 키로 노무 질문 하나를 보냅니다. 요청은 내 PC에서 VM의 서빙 API를 거쳐 GPU 파드의 vLLM으로 갑니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 발급한 키를 X-API-Key 헤더에 담아 노무 질문 하나를 보냅니다
issued_headers = {"X-API-Key": issued["api_key"]}
labor_response = httpx.post(VM_URL + "/v1/labor/chat", headers=issued_headers, json={"question": labor_questions[0]["question"]}, timeout=REQUEST_TIMEOUT)

# 성공하면 모델의 답과 토큰 수, 실패하면 error 항목의 code 와 message 를 봅니다
labor_body = labor_response.json()
if labor_response.status_code == 200:
    labor_view = {"상태 코드": 200, "답": labor_body["answer"], "usage": labor_body["usage"]}
else:
    labor_view = {"상태 코드": labor_response.status_code, "오류": labor_body["error"]["code"] + ": " + labor_body["error"]["message"]}
display(labor_view)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">200과 노무 답:</strong> 내 PC, VM, GPU 파드가 이어졌습니다. 첫 상담은 수십 초 걸릴 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">답이 중간에 끊기면:</strong> 답 길이 제한(256토큰) 때문이며 오류가 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">502 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">backend_error</code>:</strong> vLLM이 오류로 답했습니다. 파드가 Running인지, RunPod 콘솔 Logs 탭에 오류가 있는지 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">504 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">backend_timeout</code>:</strong> vLLM이 90초 안에 답하지 않았습니다.</li>
</ul>
</div>

## 6. 컨테이너를 다시 만들어도 남는 것

### 6-1. 컨테이너를 지우고 다시 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">다시 배포하면 컨테이너를 지우고 새로 만듭니다. 키 DB 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app/data</code>에 볼륨을 붙였으므로 5-3에서 발급한 키가 남아야 합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/23_volume_survives.png" width="1050" alt="docker compose down 과 up 으로 옛 컨테이너를 지우고 새 컨테이너를 만들어도 볼륨 service-data 의 키 DB는 남고, down -v 뒤에는 옛 볼륨이 지워지고 빈 service-data 볼륨이 새로 생기는 비교">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose down</code>은 컨테이너와 네트워크를 지우고 볼륨은 둡니다. 내리고 다시 띄운 뒤 볼륨 목록을 봅니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/serving-deploy/deploy
pwd   # 끝이 serving-deploy/deploy이면 맞습니다

# 컨테이너를 지우고, 띄울 이미지 태그(latest)를 정해 같은 compose.yaml 로 다시 만듭니다
docker compose down
IMAGE_TAG=latest docker compose up -d

docker volume ls    # serving-deploy_service-data 가 남아 있는지 봅니다
docker compose ps   # serving-api 의 STATUS 가 Up 인지 봅니다
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG=latest 명령</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">뒤의 그 한 명령을 실행하는 동안만 환경 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>로 정합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker volume ls</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VM의 Docker 볼륨 목록을 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Up</code>이면, 5-3의 같은 키로 사용량(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/usage</code>, 키 DB만 읽음)을 조회합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 컨테이너를 다시 만든 뒤 같은 서비스 키로 내 사용량을 조회합니다
usage_response = httpx.get(VM_URL + "/v1/usage", headers=issued_headers, timeout=REQUEST_TIMEOUT)
usage_view = {"상태 코드": usage_response.status_code, "본문": usage_response.json()}
display(usage_view)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">200, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">requests</code>에 5-4의 요청:</strong> 컨테이너를 다시 만들어도 키와 사용량 기록이 볼륨에 남았습니다.</li>
</ul>
</div>

### 6-2. 볼륨까지 지우는 down -v

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-v</code>를 붙이면 compose 파일에 선언한 볼륨까지 지웁니다. 이 교안에서는 실행하지 않습니다.</p>
</div>

```bash
docker compose down -v
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">그 뒤 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker compose up -d</code>:</strong> 빈 볼륨이 새로 생기고 서버가 빈 테이블을 만듭니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">같은 서비스 키로 요청하면:</strong> 401(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">invalid_service_key</code>)입니다. 키와 사용량 기록이 모두 사라져 키를 다시 발급해야 합니다.</li>
</ul>
</div>

## 7. GPU 파드 모니터링과 교체

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">RunPod 콘솔에서 GPU 파드가 요청을 처리하는 모습을 보고(7-1), 파드를 새로 만들어야 할 때의 순서를 익힙니다(7-2).</p>
</div>

### 7-1. GPU 사용량과 요청 로그 보기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노트북에서 5-4의 상담 셀을 다시 실행하고, 답을 기다리는 동안 콘솔에서 봅니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">RunPod 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.runpod.io/pods</code></li>
</ul>
<p style="margin:6px 0;">serving-vllm 파드를 누른 화면에서 시작합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · serving-vllm &gt; Telemetry</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> GPU 0의 VRAM(요청 없이도 약 90%)과 Utilization(요청이 없으면 0%)을 봅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> VRAM, Utilization</li></ul></div>
<img src="images/runpod_16_telemetry.png" width="1050" alt="serving-vllm > Telemetry. 빨간 상자: VRAM, Utilization">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · serving-vllm &gt; Logs &gt; Container</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Container를 고르면 vLLM 로그가 보입니다. 상담 요청마다 POST /v1/chat/completions 줄이 하나씩 남습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Container 탭, 다운로드 버튼</li></ul></div>
<img src="images/runpod_13_logs_container.png" width="1050" alt="serving-vllm > Logs > Container. 빨간 상자: Container 탭, 다운로드 버튼">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">VRAM이 요청 없이도 약 90%:</strong> vLLM이 뜰 때 GPU 메모리를 미리 잡아 KV 캐시로 예약하기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Utilization:</strong> 요청이 없으면 0%이고, 답을 만드는 동안 올라갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">&quot;POST /v1/chat/completions HTTP/1.1&quot; 200 OK</code>:</strong> 서빙 API가 보낸 상담 요청입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;GET / HTTP/1.1&quot; 404</code>는 누군가 주소 맨 앞을 부른 기록이라 오류가 아닙니다.</li>
</ul>
</div>

### 7-2. 새 GPU 파드로 바꿀 때 VM의 설정 파일 고치기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">파드가 사라지거나 다시 만들어야 하면 2-2부터 3-2까지 다시 해 새 파드를 만들고, 5-1의 셀로 준비를 기다린 뒤 아래 순서를 따릅니다.</p>
<p style="margin:6px 0;">새 파드는 id가 달라 proxy 주소가 바뀝니다. VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/.env</code>에는 옛 주소가 들어 있으므로 새 주소로 바꿉니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">내 PC <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>를 새 파드 id로 바꿉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서빙 API:</strong> 4-2 셀로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_serving.env</code>를 다시 만들고, 4-3의 scp 명령으로 VM에 올립니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">VM에서 다시 띄우기:</strong> 컨테이너를 다시 만들어야 새 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 읽습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>를 빼면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code> 이미지로 바뀌므로, 실행 중인 컨테이너의 태그를 읽어 같은 태그로 띄웁니다.</li>
</ol>
<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>
</div>

```bash
cd ~/serving-deploy/deploy
TAG=$(docker inspect --format '{{.Config.Image}}' $(docker compose ps -q serving-api) | cut -d: -f2)
IMAGE_TAG=$TAG docker compose up -d
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose ps -q serving-api</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">실행 중인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code> 컨테이너의 id를 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker inspect --format '{{.Config.Image}}' id</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 컨테이너를 만든 이미지 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/.../serving-deploy:태그</code>)을 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cut -d: -f2</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">:</code> 뒤의 태그만 남깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG=&#36;TAG docker compose up -d</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">같은 태그로 컨테이너를 다시 만들어 새 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 읽게 합니다.</td>
</tr>
</tbody>
</table>
<ol start="4">
<li style="margin:3px 0;">내 PC에서 교안 02 5-4 셀을 다시 실행해 상담이 200인지 확인합니다.</li>
</ol>
</div>